# P101 — Una reducción del aprendizaje por imitación al aprendizaje en línea sin arrepentimiento

## 1. Título y paper

**Paper:** *A Reduction of Imitation Learning and Structured Prediction to No-Regret Online Learning*  
**Autoría:** Stéphane Ross, Geoffrey J. Gordon, J. Andrew Bagnell  
**Año y venue:** 2011 · AISTATS 2011 · arXiv:1011.0686  
**Nivel:** L3 · **Motor:** `dagger`  
**Ficha completa:** [`P101_dagger`](../../papers/foundational/P101_dagger/README.md)

**Hito:** Explica por qué la clonación de comportamiento se degrada con el horizonte, y da un algoritmo que reduce el error de orden T² a orden T.

- [arXiv:1011.0686](https://arxiv.org/abs/1011.0686)

> Este notebook implementa una **miniatura** del mecanismo. No reproduce el experimento original ni sus métricas: reproduce la idea para que se pueda inspeccionar y discutir.


## 2. Objetivos

1. Explicar qué problema resolvió el paper: Al clonar el comportamiento de un experto, el modelo se entrena con los estados que visita el EXPERTO y se ejecuta sobre los estados que visita ÉL MISMO. Un error lo saca de la distribución de entrenamiento, donde comete más errores, y la desviación se realimenta.
2. Ejecutar una implementación mínima de la propuesta: DAgger: ejecutar la política actual, recoger los estados que visita de verdad, pedir al experto la acción correcta **en esos estados**, y reentrenar sobre el conjunto acumulado. La distribución de entrenamiento converge a la de ejecución.
3. Predecir el resultado antes de ejecutar, y contrastar la predicción con la salida.
4. Identificar al menos una limitación de la miniatura y una del paper original.
5. Conectar el hito con el siguiente eslabón de la ruta.


## 3. Prerrequisitos

- Python 3.11+ y el paquete del programa instalado (`pip install -e .`).
- Haber leído la guía [método de lectura en 5 pasadas](../../papers/guides/METODO_DE_LECTURA_EN_5_PASADAS.md).
- Hitos previos:
- Pomerleau (1989), ALVINN: conducción por clonación


## 4. Intuición

Clonar el comportamiento de un experto parece aprendizaje supervisado corriente: pares de estado y acción, y a entrenar. La trampa es que el modelo se entrena con los estados que visita el experto y luego se ejecuta sobre los estados que visita él mismo — y en cuanto se equivoca una vez, ya no son los mismos.


## 5. Concepto mínimo

```text
Clonación:  entrena con D ~ distribución del EXPERTO
            ejecuta sobre  distribución de la POLÍTICA

    un error → estado nuevo → sin ejemplos → más errores → más desviación
    error total ~ O(T²)   con T el horizonte

DAgger:     ejecutar la política, PREGUNTAR AL EXPERTO en los estados visitados,
            reentrenar sobre el acumulado   →   error ~ O(T)
```


## 6. Código explicado

El motor aísla el mecanismo del paper con datos de juguete y salida inspeccionable.


In [ ]:
import json
import pathlib
import sys

ROOT = pathlib.Path.cwd()
while not (ROOT / "pyproject.toml").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))

from ai_evolution.papers_lab import run_paper_lab


def show(value):
    print(json.dumps(value, ensure_ascii=False, indent=2))


In [ ]:
r = run_paper_lab('dagger', seed=7)['result']
show(r)

## 7. Predicción antes de ejecutar

1. ¿Cuántos episodios completa la clonación de comportamiento?
2. ¿Cuántos estados cubre su entrenamiento?
3. ¿Qué pasa al ampliar la cobertura con los estados que la política visita de verdad?

> Escribe tu respuesta aquí antes de continuar.


## 8. Experimento controlado

Se varía una sola cosa y se observa el efecto.


In [ ]:
for semilla in (1, 7, 42):
    r = run_paper_lab('dagger', seed=semilla)
    print(f'semilla {semilla:>2} · evidencia principal:')
    for e in r['evidence']:
        print('   +', e)
    break  # determinista: basta una para ver la estructura
for semilla in (1, 7, 42):
    r = run_paper_lab('dagger', seed=semilla)['result']
    print(f'semilla {semilla:>2} → claves: {list(r)[:4]}')

## 9. Salida interpretable

La clonación completa **264 de 300** episodios (88 %), y su cobertura son los **25 estados** del carril central: los únicos que el experto pisa. DAgger amplía la cobertura a **119 estados** y la tasa sube de 0,903 a **1,0** en cinco iteraciones.


## 10. Comentario pedagógico

El fallo no es que la política sea mala: es que fuera del carril central **no tiene ejemplos**, y ahí se equivoca más, lo que la aleja más. La desviación se realimenta. Ese mismo patrón aparece en cualquier sistema que se entrene con trayectorias de éxito y se despliegue en un bucle cerrado — incluidos los agentes con modelos de lenguaje.


## 11. Error o anti-patrón deliberado

Anti-patrón: evaluar una política clonada solo sobre las trayectorias del experto.


In [ ]:
print('Sobre los estados del experto, la clonacion acierta casi siempre: no hay novedad ahi.')
print('La evaluacion honesta es EJECUTAR la politica y medir el resultado del episodio.')
print('La diferencia entre esas dos cifras es exactamente el problema del articulo.')

## 12. Corrección

La medición correcta, ejecutando la política:


In [ ]:
r = run_paper_lab('dagger', seed=7)['result']
print('clonacion:', r['clonacion_de_comportamiento'])
for h in r['dagger_por_iteracion']:
    print(f"  iter {h['iteracion']}  exito {h['exito']}  cobertura {h['cobertura']}"
          f"  nuevos {h['estados_nuevos_etiquetados']}")

## 13. Desafío guiado

Localiza en qué iteración deja de haber estados nuevos que etiquetar, y explica qué significa eso sobre la distribución de la política.


In [ ]:
r = run_paper_lab('dagger', seed=3)['result']
show(r)

## 14. Desafío autónomo

Entrena una política por clonación sobre un entorno de control simple, mide su tasa de éxito ejecutándola, y aplica una iteración de DAgger. Documenta cuántas consultas al experto hicieron falta.


## 15. Evidencia de aprendizaje

Guarda la evolución de cobertura y tasa de éxito, con tu explicación del cambio de distribución.

Autoevaluación y respuestas esperadas: [ficha del paper](../../papers/foundational/P101_dagger/README.md) · evaluación formal: [`assessments/papers/P101_dagger.md`](../../assessments/papers/P101_dagger.md)


## 16. Cierre

Copiar a un experto exige tener uno. Cuando no lo hay, la política tiene que salir de la propia experiencia — y ahí el problema es que un solo paso malo destruya lo aprendido.


## 17. Conexión con el siguiente hito

- P102

Ruta completa: [`papers/ROADMAP.md`](../../papers/ROADMAP.md)
